# Milestone 4 lab, notebook 7 of 8: Test the tests

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| **7. Test the tests** (you are here) | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. The image has changed since milestone 1: it now holds dbt, so the first start builds for a minute or two.

**About ten minutes.**

In [1]:
import pandas as pd
from lab_helpers import MILESTONE_DIR, backup, change, prepare_week, restore, run_from, sql

import airflow_tools  # lab_helpers put this milestone's folder on the path, so this is found

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.


## Step 1: A gate you have never seen fail

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

**Dana Okafor** manages store `S02`. Her Friday file came in twice, and for a day her sales were $1,034.28 too high. A gate caught it. How do you know it will catch the next one?

A gate that has never failed is a gate you only hope works. So you test the tests: plant a fault, and watch its gate catch it.

## Step 2: The thirteen planted cases, and their gates

The stores' files hold thirteen faults. Here is every test that found something in the bonus week, with the first night it spoke.

In [2]:
sql("""
    SELECT min(run_day) AS first_night, gate, test_name, action, max(bad_rows) AS bad_rows, title
    FROM ops.gate_results
    WHERE status <> 'pass'
    GROUP BY gate, test_name, action, title
    ORDER BY min(run_day), gate, test_name
""")

,first_night,gate,test_name,action,bad_rows,title
0,2026-08-24,clean,line_adds_up,quarantine,1,A line adds up
1,2026-08-24,clean,refund_gives_money_back,quarantine,1,A refund gives money back
2,2026-08-24,publish,store_is_not_ready_to_pay,warn,5,Every open store has a day we can pay from
3,2026-08-25,clean,true_key_unique,quarantine,171,One item line is stored once
4,2026-08-25,landing,file_has_a_new_column,warn,1,A file has no column we did not expect
5,2026-08-25,landing,file_is_missing_a_column,refuse_file,1,A file has every column we expect
6,2026-08-25,landing,supplier_contract,refuse_file,2,A supplier's file keeps the contract
7,2026-08-26,clean,one_person_once,quarantine,1,One person is one member
8,2026-08-27,clean,believable_quantity,quarantine,1,A weight is a believable weight
9,2026-08-28,clean,a_count_for_every_day,warn,1,A snapshot has a row for every day


Go down the list with the planted cases in your head.

- Monday: the refund, set aside by two tests.
- Tuesday: the receipt sent twice. The supplier's renamed column, refused at landing by two tests, with the new column noticed too.
- Wednesday: the member who signed up twice.
- Thursday: the bananas in grams.
- Friday: the butter price, the file sent twice and its strange size, the milk nobody counted. And the supplier's cents: the contract test goes from one refused file to two.
- Saturday: the soup with no product.
- Sunday: the receipt stamped at 03:10.

Two cases have no line of their own: the store with no file, and the late file. They are in the publish gate's test, `store_is_not_ready_to_pay`, which spoke on Monday already.

Every case, a gate. But these are faults you were given. Now plant your own.

## Step 3: Plant a fault no row check can see

Change one sale so that it is wrong and looks right: two pieces where one was sold, with the amount worked out again. A valid quantity, a valid price, a line that adds up.

In [3]:
backup("raw.sales_lines")
change("""
    UPDATE raw.sales_lines
    SET quantity = '2.000', amount_cents = CAST(2 * CAST(unit_price_cents AS INTEGER) AS TEXT)
    WHERE line_id = (
        SELECT min(line_id) FROM raw.sales_lines
        WHERE quantity = '1.000' AND line_type = 'sale' AND _file_name = 'sales_lines_S03_2026-08-20.csv'
    )
""")
run_from("clean")

Kept a copy of raw.sales_lines.


clean:        7 clean tables rebuilt. sales_lines has 24,384 rows.


gate clean: 39 tests, 29 passed.
    set aside     171  One item line is stored once
    set aside       1  A line adds up
    set aside       3  Every sale is for a product that exists
    set aside       1  A refund gives money back
    warning         6  The price charged is the price valid that day
    set aside       4  Nothing is sold while the store is closed
    set aside       1  A weight is a believable weight
    warning         1  A store's day is near its normal day
    warning         1  A snapshot has a row for every day
    set aside       1  One person is one member
    rows set aside: 177 in sales_lines, 3 in raw_sales_lines, 1 in members
build:        staging.daily_store_sales has 208 rows. Nobody reads staging.


gate publish: 5 tests, 3 passed.
    BLOCK           1  The receipts add up to the till's own total, or we can say why not
    warning         6  Every open store has a day we can pay from
A gate blocked. The steps after it did not run, and nothing new is published.


Read what the gates said.

The **clean gate** let the planted row through: it is not in its list. No rule about one row can know that one bottle was sold and not two.

The **publish gate** blocked. Its test is the till total, on what is left: the receipts of `S03` no longer add up to its tills, no row was set aside for that day, and no file is missing. A difference nobody can explain is a fault no check knows. That is not published.

In [4]:
sql("""
    SELECT store_id, sales_date, lines_cents, till_cents, lines_cents - till_cents AS difference_cents,
           rows_set_aside, problem
    FROM prepared.store_day_status
    WHERE problem = 'unexplained'
""")

,store_id,sales_date,lines_cents,till_cents,difference_cents,rows_set_aside,problem
0,S03,2026-08-20,71681,71282,399,0,unexplained


In [5]:
restore("raw.sales_lines")

Put back: raw.sales_lines. Sunday's night ran again, and the fault is gone.


## Step 4: Plant a fault at the door

Now a fault for the landing gate. Pretend one store's sales file arrived without its `amount_cents` column: take that column out of the list of columns the file had.

In [6]:
backup("raw.file_columns")
change("""
    DELETE FROM raw.file_columns
    WHERE file_name = 'sales_lines_S03_2026-08-20.csv' AND column_name = 'amount_cents'
""")
run_from("gate-landing")

Kept a copy of raw.file_columns.


gate landing: 3 tests, 0 passed.
    file refused    2  A file has every column we expect
    warning         1  A file has no column we did not expect
    file refused    2  A supplier's file keeps the contract
    files refused: sales_lines_S03_2026-08-20.csv, supplier_prices_SUP-B_2026-08-25.csv, supplier_prices_SUP-C_2026-08-28.csv


clean:        7 clean tables rebuilt. sales_lines has 24,263 rows.


gate clean: 39 tests, 29 passed.
    set aside     171  One item line is stored once
    set aside       1  A line adds up
    set aside       3  Every sale is for a product that exists
    set aside       1  A refund gives money back
    warning         6  The price charged is the price valid that day
    set aside       4  Nothing is sold while the store is closed
    set aside       1  A weight is a believable weight
    warning         1  A store's day is near its normal day
    warning         1  A snapshot has a row for every day
    set aside       1  One person is one member
    rows set aside: 177 in sales_lines, 3 in raw_sales_lines, 1 in members
build:        staging.daily_store_sales has 207 rows. Nobody reads staging.


gate publish: 5 tests, 4 passed.
    warning         6  Every open store has a day we can pay from
publish:      daily_store_sales has 207 rows. replenishment_list has 234 rows for tomorrow. 6 store days are not ready to pay.


In [7]:
sql("""
    SELECT s.store_id, s.sales_date, s.lines_cents, s.till_cents, s.problem, s.reasons
    FROM output.store_day_status s
    WHERE s.store_id = 'S03' AND s.sales_date = DATE '2026-08-20'
""")

,store_id,sales_date,lines_cents,till_cents,problem,reasons
0,S03,2026-08-20,0,71282,file_refused,An expected column is missing: amount_cents


The landing gate refused the file. Nothing blocked: the other stores went on. And `S03`'s day says `file_refused`, with the missing column by name.

(This is a lab trick: the rows were already loaded, and only the list of columns was changed. With a real file the column would be empty too. Milestone 5 does it for real, with a file the store sends.)

In [8]:
restore("raw.file_columns")

Put back: raw.file_columns. Sunday's night ran again, and the fault is gone.


That makes four faults you planted yourself, in notebooks 5, 6 and 7. A broken stores table blocked at the clean gate. A file that was one-third bad was refused whole. A wrong quantity blocked at the publish gate. A file with a missing column was refused at landing. The production code does the same in `tests/test_gates.py`, on every change.

## Step 5: One check, four tools

At work you may not find dbt. You may find Great Expectations, or Soda, two tools made only for data quality. They are **not installed** in this project. Here is one check in all of them, so you know the shapes.

The rule: *`line_type` is `sale` or `refund`.*

**By hand, as you wrote it in milestone 1:**

```sql
SELECT * FROM clean.sales_lines
WHERE line_type IS NULL OR line_type NOT IN ('sale', 'refund')
```

**As a dbt test, in `sources.yml`:**

```yaml
- name: line_type
  data_tests:
    - accepted_values:
        arguments:
          values: ["sale", "refund"]
```

**As it would look in Great Expectations (GX Core), in Python:**

```python
gx.expectations.ExpectColumnValuesToBeInSet(column="line_type", value_set=["sale", "refund"])
```

**As it would look in Soda (SodaCL), in YAML:**

```yaml
checks for sales_lines:
  - invalid_count(line_type) = 0:
      valid values: [sale, refund]
```

The last two are written from each tool's own documentation. They were **not run** here.

| | By hand | dbt | Great Expectations | Soda |
|---|---|---|---|---|
| You write | SQL | A line of YAML, or SQL | Python | YAML |
| The rule is called | A check | A test | An expectation | A check |
| It passes when | No rows come back | No rows come back | No value is outside the set | The count is 0 |

Four tools, four spellings, one rule. And the tool did not tell you the rule. Nothing in any of them knows who reads `sales_lines`, or that a refund gives money back. **The tool changes. The reasoning does not.**

## Step 6: A fifth shape, inside Airflow

Airflow can run a check without dbt. Its `SQLCheckOperator` runs a query that returns **one row**, and passes when every value in that row is true.

This project has one such check as a lab: the refund rule, in a small DAG of its own.

In [9]:
dag_file = (MILESTONE_DIR / "dags" / "sql_check_example_dag.py").read_text()
start = dag_file.index("    refund_gives_money_back = SQLCheckOperator(")
print(dag_file[start:].rstrip())

    refund_gives_money_back = SQLCheckOperator(
        task_id="refund_gives_money_back",
        conn_id="lotus_warehouse",  # set in docker-compose.yml, as AIRFLOW_CONN_LOTUS_WAREHOUSE
        database="lotus_m4",
        sql="""
            SELECT count(*) = 0
            FROM raw.sales_lines
            WHERE line_type = 'refund' AND CAST(amount_cents AS BIGINT) >= 0
        """,
    )


In [10]:
states = airflow_tools.run_lab_dag("lotus_sql_check_example_m4")
pd.DataFrame(states)

,task,state
0,refund_gives_money_back,failed


`failed`. The query asked "is the number of bad refunds zero?", the answer was false, and the task failed. The raw table still holds the planted refund, so that is the right answer.

| | A dbt test | Airflow's `SQLCheckOperator` |
|---|---|---|
| The query returns | The bad rows | One row: true or false |
| It passes when | No row comes back | The row is true |
| The rule lives | In the dbt project | In the DAG file |
| After a failure you have | The rows, in `audit` | "It failed" |
| Can rows be set aside? | Yes | No. It only knows pass or fail |

That last line is why this project's gates run dbt tests. To set a row aside, you must know which row.

## What you now know

- A gate you have never seen fail is a gate you only hope works. Plant the fault.
- Every one of the thirteen planted cases is caught at a gate.
- A valid row with a wrong quantity passed every row check, and the till total at the publish gate still stopped it.
- The same rule can be written by hand, in dbt, in Great Expectations, in Soda, or as an Airflow SQL check. The reasoning is the same in all of them.
- A check that returns the bad rows can do more than a check that returns true or false.

**Say it in an interview:** "I tested the gates by planting faults: a broken reference table, a file with a missing column, and a valid-looking row with a wrong quantity. The last one passed every row check, and the reconciliation at the publish gate blocked it. I can write the same rule as a dbt test, a Great Expectations expectation or a Soda check. The tool is the small part."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "A starter kit for any table", and [dbt Tests: Generic, Singular & Custom](https://www.d8loop.com/modules/knowledge-corner/dbt/dbt-tests-generic-singular-and-custom).

Next: [notebook 8](08_the_production_version.ipynb). The whole week, through Airflow.